# Model 1: Predictive Content of Congressional Trading

## Research Question
Do congressional trades contain information that improves out-of-sample prediction of stock returns beyond publicly available market information?

## Methodology

**Target Variable:** Cumulative Abnormal Return (CAR) adjusted by Fama-French 3 factors, 30-day horizon.

**Experimental Design:**
- **Model Base:** $E[CAR_{i,t+1}] = f(X^{market}_{i,t})$
- **Model Augmented:** $E[CAR_{i,t+1}] = f(X^{market}_{i,t}, X^{congress}_{i,t})$
- **Test:** If $R^2_{OOS,augmented} > R^2_{OOS,base}$ → Congressional trades contain private information

**Evaluation Framework:**
- Out-of-sample $R^2$ (Campbell & Thompson, 2008)
- Expanding window cross-validation (no look-ahead bias)
- Clark-West (2007) test for nested model comparison
- Diebold-Mariano (1995) test for forecast comparison

**References:**
- Campbell, J. Y., & Thompson, S. B. (2008). Predicting excess stock returns out of sample. *Review of Financial Studies*, 21(4), 1509-1531.
- Clark, T. E., & West, K. D. (2007). Approximately normal tests for equal predictive accuracy in nested models. *Journal of Econometrics*, 138(1), 291-311.
- Gu, S., Kelly, B., & Xiu, D. (2020). Empirical asset pricing via machine learning. *Review of Financial Studies*, 33(5), 2223-2273.
- Grossman, S. J., & Stiglitz, J. E. (1980). On the impossibility of informationally efficient markets. *American Economic Review*, 70(3), 393-408.

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

# ML
from sklearn.linear_model import (
    LinearRegression, 
    Ridge, 
    Lasso, 
    ElasticNet,
    RidgeCV,
    LassoCV,
    ElasticNetCV
)
from sklearn.ensemble import (
    RandomForestRegressor, 
    GradientBoostingRegressor
)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, r2_score
from scipy import stats

# Plotting
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Setup complete')

In [ ]:
# Figure style (paper-ready, no titles per request)
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})

# Colors (colorblind-friendly)
C_BASE = '#2166AC'       # Blue - Base model
C_AUGMENTED = '#B2182B'  # Red - Augmented model  
C_NEUTRAL = '#666666'    # Gray
C_POSITIVE = '#1A9850'   # Green
C_NEGATIVE = '#D73027'   # Red

# Output
OUTPUT_DIR = 'outputs/Model_1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Output directory: {OUTPUT_DIR}')

## 1. Load Data

In [ ]:
# Load panel
panel = pd.read_parquet('data/prediction_bases/panel_final_stock_month.parquet')

# Temporal ordering
panel['month_dt'] = pd.to_datetime(panel['month'].astype(str))
panel = panel.sort_values(['month_dt', 'ticker']).reset_index(drop=True)

print(f'Observations: {len(panel):,}')
print(f'Variables: {len(panel.columns)}')
print(f'Period: {panel["month"].min()} to {panel["month"].max()}')
print(f'Unique stocks: {panel["ticker"].nunique():,}')
print(f'Unique months: {panel["month"].nunique()}')

## 2. Target Variable

We use **Cumulative Abnormal Return (CAR)** adjusted by Fama-French 3 factors rather than raw returns.

**Rationale (Grossman-Stiglitz, 1980):**
- Raw returns include systematic components (market, size, value) that are publicly known
- CAR isolates the idiosyncratic component—the part not explained by public risk factors
- If congressmen possess private information, it should predict the abnormal component

In [ ]:
# Define target
TARGET = 'ret_future_car30_ff3'

# Check availability
if TARGET not in panel.columns:
    # Fallback options
    alternatives = ['ret_future_car30', 'ret_future_1m']
    for alt in alternatives:
        if alt in panel.columns:
            TARGET = alt
            print(f'Using alternative target: {TARGET}')
            break

print(f'Target: {TARGET}')
print(f'  N non-null: {panel[TARGET].notna().sum():,}')
print(f'  Mean: {panel[TARGET].mean():.4f}')
print(f'  Std:  {panel[TARGET].std():.4f}')
print(f'  Min:  {panel[TARGET].min():.4f}')
print(f'  Max:  {panel[TARGET].max():.4f}')

In [ ]:
# Winsorize target at 1%/99% to reduce outlier influence
lower, upper = panel[TARGET].quantile([0.01, 0.99])
panel[TARGET] = panel[TARGET].clip(lower=lower, upper=upper)

print(f'Target winsorized to [{lower:.4f}, {upper:.4f}]')

## 3. Feature Engineering

### 3.1 Market Features (Public Information)

Based on established asset pricing literature:
- **Momentum:** Jegadeesh & Titman (1993)
- **Volatility:** Ang et al. (2006)
- **Liquidity:** Amihud (2002)
- **Factor exposures:** Fama & French (1993)
- **Valuation:** Fama & French (1992)

In [ ]:
# Define market features (all ex-ante, publicly available)
FEATURES_MARKET_CANDIDATES = [
    # Momentum
    'mkt_momentum_5d',
    'mkt_momentum_20d',
    'mkt_momentum_60d',
    'mkt_momentum_252d',
    
    # Volatility
    'mkt_realized_vol_30d',
    'mkt_realized_vol_60d',
    'mkt_realized_vol_252d',
    'mkt_parkinson_vol_30d',
    
    # Liquidity
    'mkt_amihud_illiq_20d',
    'mkt_volume_ratio_30d',
    'mkt_abnormal_volume_30d',
    'mkt_roll_spread_30d',
    
    # Beta and factor exposures
    'mkt_beta_252d',
    'mkt_beta_smb_ff3_252d',
    'mkt_beta_hml_ff3_252d',
    'mkt_r2_ff3_252d',
    
    # Valuation
    'mkt_price_to_book',
    'mkt_ev_to_ebitda',
    'mkt_market_cap',
]

# Filter to available columns
FEATURES_MARKET = [f for f in FEATURES_MARKET_CANDIDATES if f in panel.columns]

print(f'Market features available: {len(FEATURES_MARKET)}/{len(FEATURES_MARKET_CANDIDATES)}')
print('Features:', FEATURES_MARKET)

### 3.2 Congressional Features (Our Contribution)

Selected based on theoretical relevance to the information hypothesis:

1. **Direction:** Net trading signal
2. **Information access:** Committee membership, chair status
3. **Political power:** Seniority, chamber
4. **Timing:** Disclosure delay
5. **Coordination:** Multiple politicians trading together
6. **Market context:** Contrarian behavior, volatility timing
7. **Composite signals:** Smart money, insider ring

In [ ]:
# Define congressional features by category
FEATURES_CONGRESS_CANDIDATES = {
    'direction': [
        'cong_csi',                    # Congressional Sentiment Index [-1, 1]
        'cong_buy_ratio',              # Proportion of buys [0, 1]
        'cong_net',                    # Net buys - sells
    ],
    'information_access': [
        'cong_info_ratio',             # % from informationally-sensitive committees
        'cong_chair_ratio',            # % from chairs/ranking members
    ],
    'power': [
        'cong_avg_power_index',        # Composite power measure [0, 4]
        'cong_senator_ratio',          # % from senators
        'cong_senior_ratio',           # % from senior members (10+ years)
    ],
    'timing': [
        'cong_avg_disclosure_delay',   # Days to disclosure
        'cong_long_delay_ratio',       # % with delay > 30 days
    ],
    'coordination': [
        'cong_coordinated_ratio',      # % coordinated (≥2 politicians same day)
        'cong_committee_coordinated_ratio',  # % same committee coordinated
        'cong_unique_politicians',     # Number of distinct politicians
    ],
    'context': [
        'cong_contrarian_ratio',       # % contrarian trades
        'cong_high_vol_ratio',         # % during high volatility
        'cong_illiquid_ratio',         # % in illiquid stocks
        'cong_small_cap_ratio',        # % in small caps
    ],
    'composite': [
        'cong_smart_money',            # Chair + info committee + buying
        'cong_has_insider_ring',       # Committee coordination in info committee
        'cong_has_hidden',             # Illiquid/small + info committee
        'cong_strong_buy',             # CSI > 0.5 + multiple politicians
        'cong_consensus_buy',          # Buy ratio > 70%
    ],
    'intensity': [
        'cong_total_trades',           # Total trades
        'cong_large_ratio',            # % large trades (≥$100K)
        'cong_intensity',              # Trades per politician
    ],
}

# Flatten and filter to available
FEATURES_CONGRESS_ALL = []
for category, features in FEATURES_CONGRESS_CANDIDATES.items():
    available = [f for f in features if f in panel.columns]
    FEATURES_CONGRESS_ALL.extend(available)
    print(f'{category}: {len(available)}/{len(features)} available')

print(f'\nTotal congressional features: {len(FEATURES_CONGRESS_ALL)}')

In [ ]:
# Core congressional features (parsimonious set for main analysis)
FEATURES_CONGRESS_CORE_CANDIDATES = [
    'cong_csi',
    'cong_info_ratio',
    'cong_chair_ratio',
    'cong_avg_power_index',
    'cong_avg_disclosure_delay',
    'cong_coordinated_ratio',
    'cong_committee_coordinated_ratio',
    'cong_contrarian_ratio',
    'cong_smart_money',
    'cong_unique_politicians',
    'cong_total_trades',
]

FEATURES_CONGRESS_CORE = [f for f in FEATURES_CONGRESS_CORE_CANDIDATES if f in panel.columns]
print(f'Core congressional features: {len(FEATURES_CONGRESS_CORE)}')

In [ ]:
# Define feature sets for comparison
FEATURE_SETS = {
    'Base (Market)': FEATURES_MARKET,
    'Augmented (Market + Congress Core)': FEATURES_MARKET + FEATURES_CONGRESS_CORE,
    'Full (Market + All Congress)': FEATURES_MARKET + FEATURES_CONGRESS_ALL,
}

print('Feature sets:')
for name, features in FEATURE_SETS.items():
    print(f'  {name}: {len(features)} features')

### 3.3 Preprocessing

In [ ]:
# Combine all features
ALL_FEATURES = list(set(FEATURES_MARKET + FEATURES_CONGRESS_ALL))

# Handle missing values
print('Missing values before cleaning:')
missing_before = panel[ALL_FEATURES].isnull().sum().sum()
print(f'  Total NaN: {missing_before:,}')

# Fill NaN with median (conservative approach)
for col in ALL_FEATURES:
    if panel[col].isnull().any():
        panel[col] = panel[col].fillna(panel[col].median())

# Winsorize features at 1%/99%
for col in ALL_FEATURES:
    lower, upper = panel[col].quantile([0.01, 0.99])
    panel[col] = panel[col].clip(lower=lower, upper=upper)

# Drop rows with missing target
panel = panel.dropna(subset=[TARGET])

print(f'\nFinal sample: {len(panel):,} observations')
print(f'Missing values after cleaning: {panel[ALL_FEATURES].isnull().sum().sum()}')

## 4. Evaluation Framework

### 4.1 Out-of-Sample $R^2$

Following Campbell & Thompson (2008):

$$R^2_{OOS} = 1 - \frac{\sum_{t=1}^{T} (r_t - \hat{r}_t)^2}{\sum_{t=1}^{T} (r_t - \bar{r}_t)^2}$$

where $\bar{r}_t$ is the expanding-window historical mean (benchmark).

**Interpretation:**
- $R^2_{OOS} > 0$: Model beats historical mean
- $R^2_{OOS} = 0$: Model equals historical mean
- $R^2_{OOS} < 0$: Model worse than historical mean

In [ ]:
def oos_r2(y_true, y_pred, y_benchmark):
    """
    Out-of-sample R² (Campbell & Thompson, 2008).
    
    Parameters:
    -----------
    y_true : array-like
        Actual values
    y_pred : array-like
        Model predictions
    y_benchmark : array-like
        Benchmark predictions (historical mean)
    
    Returns:
    --------
    float : OOS R²
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_benchmark = np.array(y_benchmark)
    
    sse_model = np.sum((y_true - y_pred) ** 2)
    sse_benchmark = np.sum((y_true - y_benchmark) ** 2)
    
    if sse_benchmark == 0:
        return np.nan
    
    return 1 - (sse_model / sse_benchmark)

### 4.2 Statistical Tests

**Clark-West (2007)** test for nested models:
- $H_0$: Restricted model (Base) is sufficient
- $H_1$: Unrestricted model (Augmented) has superior predictive ability

**Diebold-Mariano (1995)** test:
- $H_0$: Equal predictive accuracy
- $H_1$: Unequal predictive accuracy

In [ ]:
def clark_west_test(y_true, pred_restricted, pred_unrestricted):
    """
    Clark-West (2007) test for nested models.
    
    Tests whether unrestricted model significantly outperforms
    the restricted model, adjusting for parameter estimation error.
    
    Returns:
    --------
    tuple : (statistic, p-value)
    """
    y_true = np.array(y_true)
    pred_r = np.array(pred_restricted)
    pred_u = np.array(pred_unrestricted)
    
    e1 = y_true - pred_r  # Restricted model errors
    e2 = y_true - pred_u  # Unrestricted model errors
    
    # MSPE-adjusted statistic
    adj = (pred_r - pred_u) ** 2
    f = e1**2 - (e2**2 - adj)
    
    # t-statistic
    n = len(f)
    t_stat = np.mean(f) / (np.std(f, ddof=1) / np.sqrt(n))
    
    # One-sided p-value (H1: unrestricted is better)
    p_value = 1 - stats.norm.cdf(t_stat)
    
    return t_stat, p_value


def diebold_mariano_test(e1, e2, h=1):
    """
    Diebold-Mariano (1995) test for equal predictive accuracy.
    
    Parameters:
    -----------
    e1 : array-like
        Forecast errors from model 1
    e2 : array-like
        Forecast errors from model 2
    h : int
        Forecast horizon
    
    Returns:
    --------
    tuple : (statistic, p-value)
    """
    e1 = np.array(e1)
    e2 = np.array(e2)
    
    d = e1**2 - e2**2  # Loss differential
    
    n = len(d)
    d_mean = np.mean(d)
    d_var = np.var(d, ddof=1)
    
    if d_var == 0:
        return 0, 1
    
    # Harvey-Leybourne-Newbold (1997) small-sample correction
    dm_stat = d_mean / np.sqrt(d_var / n)
    
    # Two-sided p-value
    p_value = 2 * (1 - stats.norm.cdf(abs(dm_stat)))
    
    return dm_stat, p_value

### 4.3 Expanding Window Cross-Validation

To prevent look-ahead bias, we use expanding window estimation:

At each period $t$:
1. Train on all observations from periods $[1, t-1]$
2. Predict all observations in period $t$
3. Store predictions and actuals
4. Advance to $t+1$

In [ ]:
class ExpandingWindowCV:
    """
    Expanding window cross-validation for panel data.
    
    Ensures no future information is used in training.
    """
    
    def __init__(self, min_train_periods=36):
        """
        Parameters:
        -----------
        min_train_periods : int
            Minimum number of months for initial training window
        """
        self.min_train = min_train_periods
    
    def generate_predictions(self, data, target, features, model, scale=True):
        """
        Generate out-of-sample predictions using expanding window.
        
        Returns:
        --------
        tuple : (predictions, actuals, dates)
        """
        months = sorted(data['month_dt'].unique())
        
        if len(months) <= self.min_train:
            raise ValueError(f'Insufficient data: {len(months)} months < {self.min_train} minimum')
        
        all_preds = []
        all_actuals = []
        all_dates = []
        
        for t in range(self.min_train, len(months)):
            # Define train/test split
            train_months = months[:t]
            test_month = months[t]
            
            train_mask = data['month_dt'].isin(train_months)
            test_mask = data['month_dt'] == test_month
            
            X_train = data.loc[train_mask, features].values
            y_train = data.loc[train_mask, target].values
            X_test = data.loc[test_mask, features].values
            y_test = data.loc[test_mask, target].values
            
            if len(X_test) == 0:
                continue
            
            # Scale features (fit on train only)
            if scale:
                scaler = StandardScaler()
                X_train = scaler.fit_transform(X_train)
                X_test = scaler.transform(X_test)
            
            # Fit and predict
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            
            all_preds.extend(preds)
            all_actuals.extend(y_test)
            all_dates.extend([test_month] * len(y_test))
        
        return np.array(all_preds), np.array(all_actuals), all_dates
    
    def historical_mean_benchmark(self, data, target):
        """
        Generate expanding-window historical mean predictions.
        """
        months = sorted(data['month_dt'].unique())
        
        all_preds = []
        all_actuals = []
        
        for t in range(self.min_train, len(months)):
            train_months = months[:t]
            test_month = months[t]
            
            train_mask = data['month_dt'].isin(train_months)
            test_mask = data['month_dt'] == test_month
            
            # Historical mean from training data
            hist_mean = data.loc[train_mask, target].mean()
            y_test = data.loc[test_mask, target].values
            
            all_preds.extend([hist_mean] * len(y_test))
            all_actuals.extend(y_test)
        
        return np.array(all_preds), np.array(all_actuals)

## 5. Model Specifications

Following Gu, Kelly & Xiu (2020), we use multiple algorithms:

1. **OLS:** Benchmark linear model
2. **Ridge:** L2 regularization (Hoerl & Kennard, 1970)
3. **LASSO:** L1 regularization with feature selection (Tibshirani, 1996)
4. **Elastic Net:** Combined L1/L2 (Zou & Hastie, 2005)
5. **Random Forest:** Non-linear ensemble (Breiman, 2001)
6. **Gradient Boosting:** Sequential ensemble (Friedman, 2001)

All ensemble methods use conservative hyperparameters to prevent overfitting.

In [ ]:
# Regularization grid for linear models
ALPHAS = np.logspace(-4, 2, 50)
L1_RATIOS = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95]

# Model specifications
MODELS = {
    # Linear models
    'OLS': LinearRegression(),
    
    'Ridge': RidgeCV(
        alphas=ALPHAS,
        cv=5,
        scoring='neg_mean_squared_error'
    ),
    
    'LASSO': LassoCV(
        alphas=ALPHAS,
        cv=5,
        max_iter=10000,
        random_state=RANDOM_STATE
    ),
    
    'ElasticNet': ElasticNetCV(
        l1_ratio=L1_RATIOS,
        alphas=ALPHAS,
        cv=5,
        max_iter=10000,
        random_state=RANDOM_STATE
    ),
    
    # Ensemble models (conservative hyperparameters)
    'RandomForest': RandomForestRegressor(
        n_estimators=200,
        max_depth=3,           # Shallow trees
        min_samples_leaf=50,   # Large leaf size
        max_features='sqrt',   # Feature subsample
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=100,
        max_depth=2,           # Very shallow
        learning_rate=0.01,    # Slow learning
        min_samples_leaf=50,
        subsample=0.8,         # Row subsample
        random_state=RANDOM_STATE
    ),
}

print(f'Models: {list(MODELS.keys())}')

## 6. Main Analysis

In [ ]:
# Initialize cross-validator
MIN_TRAIN_MONTHS = 36  # 3 years minimum training
cv = ExpandingWindowCV(min_train_periods=MIN_TRAIN_MONTHS)

# Generate benchmark predictions
print('Generating benchmark (historical mean)...')
bench_pred, bench_actual = cv.historical_mean_benchmark(panel, TARGET)
print(f'Out-of-sample predictions: {len(bench_pred):,}')

In [ ]:
# Run comparison: Base vs Augmented (Core)
COMPARISON_SETS = {
    'Base': FEATURES_MARKET,
    'Augmented': FEATURES_MARKET + FEATURES_CONGRESS_CORE,
}

# Store results
results = []
predictions = {'benchmark': bench_pred}

print('\n' + '='*70)
print('MODEL ESTIMATION')
print('='*70)

for feat_name, features in COMPARISON_SETS.items():
    print(f'\n--- {feat_name} ({len(features)} features) ---')
    
    for model_name, model in MODELS.items():
        print(f'  {model_name}...', end=' ', flush=True)
        
        try:
            # Generate OOS predictions
            pred, actual, dates = cv.generate_predictions(
                panel, TARGET, features, model, scale=True
            )
            
            # Align with benchmark
            n = min(len(pred), len(bench_pred))
            pred = pred[:n]
            actual = actual[:n]
            bench = bench_pred[:n]
            
            # Store predictions
            predictions[f'{model_name}_{feat_name}'] = pred
            
            # Compute metrics
            r2 = oos_r2(actual, pred, bench)
            mse = mean_squared_error(actual, pred)
            corr = np.corrcoef(actual, pred)[0, 1]
            
            results.append({
                'Model': model_name,
                'Features': feat_name,
                'N_features': len(features),
                'N_obs': n,
                'R2_OOS': r2,
                'R2_OOS_pct': r2 * 100,
                'MSE': mse,
                'RMSE': np.sqrt(mse),
                'Correlation': corr,
            })
            
            print(f'R²={r2*100:+.4f}%')
            
        except Exception as e:
            print(f'ERROR: {e}')

results_df = pd.DataFrame(results)
print(f'\nCompleted: {len(results_df)} model-feature combinations')

## 7. Results

In [ ]:
# Create pivot table
pivot_r2 = results_df.pivot(index='Model', columns='Features', values='R2_OOS_pct')
pivot_r2['Δ (pp)'] = pivot_r2['Augmented'] - pivot_r2['Base']
pivot_r2 = pivot_r2.sort_values('Δ (pp)', ascending=False)

print('='*60)
print('OUT-OF-SAMPLE R² (%) BY MODEL')
print('='*60)
print(pivot_r2.round(4).to_string())

In [ ]:
# Clark-West tests
print('\n' + '='*60)
print('CLARK-WEST TEST (2007)')
print('H0: Base model sufficient | H1: Augmented improves prediction')
print('='*60)

cw_results = []
actual_aligned = bench_actual[:len(bench_pred)]

for model_name in MODELS.keys():
    key_base = f'{model_name}_Base'
    key_aug = f'{model_name}_Augmented'
    
    if key_base in predictions and key_aug in predictions:
        pred_base = predictions[key_base]
        pred_aug = predictions[key_aug]
        
        n = min(len(pred_base), len(pred_aug), len(actual_aligned))
        
        cw_stat, cw_pval = clark_west_test(
            actual_aligned[:n], 
            pred_base[:n], 
            pred_aug[:n]
        )
        
        sig = '***' if cw_pval < 0.01 else '**' if cw_pval < 0.05 else '*' if cw_pval < 0.10 else ''
        
        cw_results.append({
            'Model': model_name,
            'CW_stat': cw_stat,
            'p_value': cw_pval,
            'Significant': sig
        })
        
        print(f'{model_name:18s}  CW = {cw_stat:+7.3f}  p = {cw_pval:.4f} {sig}')

cw_df = pd.DataFrame(cw_results)
print('\nSignificance: *** p<0.01, ** p<0.05, * p<0.10')

In [ ]:
# Correlation pivot
pivot_corr = results_df.pivot(index='Model', columns='Features', values='Correlation')
pivot_corr['Δ'] = pivot_corr['Augmented'] - pivot_corr['Base']
pivot_corr = pivot_corr.sort_values('Δ', ascending=False)

print('\n' + '='*60)
print('PREDICTION CORRELATION')
print('='*60)
print(pivot_corr.round(4).to_string())

## 8. Figures

In [ ]:
# Figure 1: R² comparison
fig, ax = plt.subplots(figsize=(8, 4.5))

models = pivot_r2.index.tolist()
x = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, pivot_r2['Base'], width, 
               label='Base (Market)', color=C_BASE, edgecolor='white', linewidth=0.5)
bars2 = ax.bar(x + width/2, pivot_r2['Augmented'], width,
               label='Augmented (+Congress)', color=C_AUGMENTED, edgecolor='white', linewidth=0.5)

ax.axhline(0, color=C_NEUTRAL, linewidth=0.8)
ax.set_ylabel('Out-of-Sample R² (%)')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=45, ha='right')
ax.legend(frameon=False, loc='upper right')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig1_r2_comparison.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig1_r2_comparison.png')
plt.show()

In [ ]:
# Figure 2: Improvement by model
fig, ax = plt.subplots(figsize=(6, 4.5))

improvements = pivot_r2['Δ (pp)'].sort_values()
colors = [C_POSITIVE if v > 0 else C_NEGATIVE for v in improvements]

bars = ax.barh(improvements.index, improvements.values, 
               color=colors, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Δ R² OOS (percentage points)')

# Add value labels
for bar, val in zip(bars, improvements.values):
    x_pos = val + 0.005 if val >= 0 else val - 0.005
    ha = 'left' if val >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2, f'{val:+.4f}',
            va='center', ha=ha, fontsize=8)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig2_improvement.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig2_improvement.png')
plt.show()

In [ ]:
# Figure 3: Clark-West statistics
fig, ax = plt.subplots(figsize=(6, 4))

cw_sorted = cw_df.sort_values('CW_stat')
colors = [C_AUGMENTED if p < 0.10 else C_NEUTRAL for p in cw_sorted['p_value']]

bars = ax.barh(cw_sorted['Model'], cw_sorted['CW_stat'],
               color=colors, edgecolor='white', linewidth=0.5)

# Critical values
ax.axvline(1.28, color=C_NEUTRAL, linewidth=0.8, linestyle='--', alpha=0.7, label='10% critical')
ax.axvline(1.64, color=C_NEUTRAL, linewidth=0.8, linestyle='-', alpha=0.7, label='5% critical')
ax.axvline(0, color='black', linewidth=0.8)

ax.set_xlabel('Clark-West Statistic')
ax.legend(frameon=False, loc='lower right', fontsize=8)

# Add significance markers
for bar, (_, row) in zip(bars, cw_sorted.iterrows()):
    if row['Significant']:
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                row['Significant'], va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig3_clark_west.pdf')
plt.savefig(f'{OUTPUT_DIR}/fig3_clark_west.png')
plt.show()

## 9. Export Results

In [ ]:
# Save results
results_df.to_csv(f'{OUTPUT_DIR}/results_full.csv', index=False)
pivot_r2.to_csv(f'{OUTPUT_DIR}/results_r2_pivot.csv')
cw_df.to_csv(f'{OUTPUT_DIR}/clark_west_tests.csv', index=False)

# Save feature sets
feature_sets_export = {
    'market': FEATURES_MARKET,
    'congress_core': FEATURES_CONGRESS_CORE,
    'congress_all': FEATURES_CONGRESS_ALL,
}
with open(f'{OUTPUT_DIR}/feature_sets.json', 'w') as f:
    json.dump(feature_sets_export, f, indent=2)

print(f'Results saved to {OUTPUT_DIR}/')

## 10. Summary

In [ ]:
# Summary statistics
n_improve = (pivot_r2['Δ (pp)'] > 0).sum()
n_total = len(pivot_r2)
avg_improvement = pivot_r2['Δ (pp)'].mean()
n_sig = (cw_df['p_value'] < 0.10).sum() if len(cw_df) > 0 else 0

best_base = results_df[results_df['Features'] == 'Base'].nlargest(1, 'R2_OOS').iloc[0]
best_aug = results_df[results_df['Features'] == 'Augmented'].nlargest(1, 'R2_OOS').iloc[0]

print('='*70)
print('SUMMARY')
print('='*70)
print(f'''
DATA:
  Total observations:     {len(panel):,}
  OOS predictions:        {len(bench_pred):,}
  Training window:        {MIN_TRAIN_MONTHS} months minimum
  Target:                 {TARGET}

FEATURES:
  Market (Base):          {len(FEATURES_MARKET)}
  Congressional (Core):   {len(FEATURES_CONGRESS_CORE)}
  Total (Augmented):      {len(FEATURES_MARKET) + len(FEATURES_CONGRESS_CORE)}

BEST MODELS:
  Base:      {best_base['Model']:18s} R²_OOS = {best_base['R2_OOS_pct']:+.4f}%
  Augmented: {best_aug['Model']:18s} R²_OOS = {best_aug['R2_OOS_pct']:+.4f}%

COMPARISON:
  Models improved by Congress:     {n_improve}/{n_total}
  Average improvement:             {avg_improvement:+.4f} pp
  Clark-West significant (p<0.10): {n_sig}/{n_total}
''')

# Interpretation
print('INTERPRETATION:')
if avg_improvement > 0 and n_improve > n_total / 2:
    print('  Congressional trading features improve out-of-sample prediction')
    print('  of abnormal returns in the majority of models.')
    if n_sig > 0:
        print(f'  {n_sig} models show statistically significant improvement (Clark-West, p<0.10).')
    print('\n  This is consistent with Grossman-Stiglitz (1980):')
    print('  congressional trades contain information not fully reflected in prices.')
elif avg_improvement > 0:
    print('  Mixed evidence: some models improve, but not a majority.')
else:
    print('  No evidence that congressional features improve prediction.')

print('\nCAVEATS:')
print('  - Predictive power ≠ causal effect (Mullainathan & Spiess, 2017)')
print('  - Results may vary across time periods')
print('  - Feature importance analysis needed to identify drivers')

print('\n' + '='*70)